**Act 1 to Act 2.** Act 1 gave us the panel and the baseline. On the 9-target discovery panel, GBSA-locked (`igb2_di4_salt0.15_st0.0072`) scores BEDROC α=20 (early-enrichment metric, top 5% of the list) = **0.541** with 4A5S imputed at 0. Docking sits at 0.516. The two overlap within 95% CIs.

Act 2 asks one thing. Is that GBSA lock stable to physics-parameter and MD-runtime tweaks? If GBSA's own choice-of-parameters variance is large, any MD feature that "beats" it might just be beating noise we underestimated.

NBs 07 to 13 walk through it. Physics importance (07). MM-GBSA and MD DOE (08, design-of-experiments toolkit). Taguchi L27 GROMACS screen (09, orthogonal 27-run array over 5 factors). Stability under timestep and throughput (10 to 11). Temporal convergence (12). LOTO (leave-one-target-out) selection correction that pins how much active-count imbalance inflates BEDROC (13).

Skip to 14 if you only care about MD signal. The CI width you see in Act 4 is set here.


> **Reader guide.** *Experiment A2 (see [STUDY_DESIGN §A2](../../STUDY_DESIGN.md)):* GBSA
> full-factorial variance decomposition — can we cheapen the physics without losing ranking?
>
> **Question:** *of the 48 GBSA combos (igb × intdiel × saltcon × surften), which factor
> dominates per-target ranking variance, and can factors with low η² be dropped for cheaper
> future runs?*
>
> **Method:** one-way η² decomposition of Kendall τ + BEDROC α=20 per combo factor, per target
> and averaged over the discovery panel.
>
> **Reproducibility contract:** reads `data/raw/reference/ohds_gbsa_dG_raw.csv` +
> `data/raw/reference/ohds_metadata.csv`; every plot has its underlying η² table under
> `data/derived/`.

# 07 — Physics-factor importance (variance decomposition)

> **Discovery stage — hypothesis-generating, not confirmatory.** Discovery-9 (9 targets × 30 measured ligands, no decoys) is used to *choose* the GBSA scoring combo and to *generate* the MM-GBSA-beats-docking hypothesis. Any significance here is subject to combo selection (winner's curse). Treat it as a trend. The confirmatory claim is deferred to the pre-registered locked **n=18** validation (`VALIDATION_PLAN.md`).

See `docs/GLOSSARY.md` for term definitions (LOTO, BEDROC α=20, sp-config, Taguchi L27, P1/P2/P3, BH-FDR, hardening, canonical baseline, panel).


In [ ]:
NB_STEM = "20_physics_importance"
# ===== repo-relative setup — reruns from a fresh clone, no absolute paths =====
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.axes import Axes
from matplotlib.figure import Figure
from scipy import stats
from IPython.display import display

# make the in-repo package importable even without `pip install -e` (fresh clone)
_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / "pyproject.toml").is_file()), _here)
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

from gbsabench.paths import RAW, DERIVED, FIGURES, TABLES
from gbsabench import style, metrics
from gbsabench.io import load
style.apply_style()
NAVY, GOLD, GREY, GREYD, CREAM, WHITE = style.NAVY, style.GOLD, style.GREY, style.GREY_DASH, style.CREAM, style.WHITE

# Publication style: in-figure titles are suppressed. The premise -- "markdown + captions
# carry the description" -- was FALSE: no caption file existed, so 18 set_title calls were
# silently deleted across the package, including five panel labels in the main deliverable
# figure and the word PARTIAL on the partial-coverage map (referee finding, rounds 6-7).
#
# Titles are still suppressed for publication, but they are now RECORDED rather than
# discarded, and figures/CAPTIONS.md is generated from what was captured -- so the
# description really does exist somewhere a reader can reach.
_SUPPRESSED_TITLES = []
def _capture_title(self, *a, **k):
    if a and isinstance(a[0], str) and a[0].strip():
        _SUPPRESSED_TITLES.append((NB_STEM, a[0].strip()))
def _capture_suptitle(self, *a, **k):
    if a and isinstance(a[0], str) and a[0].strip():
        _SUPPRESSED_TITLES.append((NB_STEM, a[0].strip()))
Axes.set_title  = _capture_title
Figure.suptitle = _capture_suptitle

# capture every figure as it is created, so the last cell can export them all to figures/
_CREATED_FIGS = []
if not getattr(plt.subplots, "_gbsa_wrapped", False):   # idempotent: never re-wrap on a dirty kernel
    _orig_subplots, _orig_figure = plt.subplots, plt.figure
    def _register(fig):
        # plt.subplots() calls plt.figure() internally, so a figure made with subplots
        # hit BOTH wrappers and was captured twice -- which is why every notebook
        # exported byte-identical fig1/fig2 pairs (referee finding, iteration 1).
        if not any(fig is seen for seen in _CREATED_FIGS):
            _CREATED_FIGS.append(fig)
        return fig
    def _cap_subplots(*a, **k):
        result = _orig_subplots(*a, **k); _register(result[0]); return result
    def _cap_figure(*a, **k):
        return _register(_orig_figure(*a, **k))
    _cap_subplots._gbsa_wrapped = _cap_figure._gbsa_wrapped = True
    plt.subplots, plt.figure = _cap_subplots, _cap_figure

# reviewer-friendly TABLE HEADERS (display only; the raw short column names stay unchanged underneath)
FRIENDLY = {
    "target": "target", "n": "ligands", "actives": "actives", "inactives": "inactives",
    "total": "ligands (total)", "gbsa_measured": "measured", "gbsa_pct": "measured %",
    "gbsa_actives": "actives", "gbsa_inactives": "inactives",
    "tau_gbsa": "Kendall τ (GBSA)", "tau_dock": "Kendall τ (dock)",
    "bedroc_gbsa": "BEDROC (GBSA)", "bedroc_dock": "BEDROC (dock)",
    "auc_gbsa": "ROC-AUC (GBSA)", "auc_dock": "ROC-AUC (dock)", "delong_p": "DeLong p",
    "metric": "metric", "gbsa_median": "GBSA (median)", "dock_median": "dock (median)",
    "wins": "GBSA wins", "p_onesided": "one-sided p", "p_twosided": "two-sided p",
    "quantity": "quantity", "value": "value", "role": "role", "description": "description",
    "factor": "factor", "eta2_tau": "η² on τ", "eta2_bedroc": "η² on BEDROC",
    "eta2_bedroc_target_blocked": "η² on BEDROC (target-blocked)",
    "eta2_nsday_MECHANICAL": "η² on ns/day (mechanical)", "eta2_steps_per_sec_REAL": "η² on steps/sec (real)",
    "dt_fs": "dt (fs)", "attempts": "attempts", "ok": "succeeded", "fail_rate_pct": "fail-rate %",
    "median_nsday_OK": "median ns/day (OK runs)", "usable_nsday_after_failures": "usable ns/GPU-day",
    "estimator": "estimator", "p_with_4L7G": "p (with 4L7G)", "p_without_4L7G": "p (without 4L7G)",
    "pdb": "PDB", "family": "family", "crystal_ligand": "crystal ligand",
    "crystal_ligand_rscc": "RSCC", "resolution_A": "resolution (Å)", "split": "split",
}
if not getattr(pd.DataFrame._repr_html_, "_gbsa_wrapped", False):   # idempotent: never re-wrap on a dirty kernel
    _orig_html, _orig_repr = pd.DataFrame._repr_html_, pd.DataFrame.__repr__
    def _friendly_html(self): return _orig_html(self.rename(columns={c: FRIENDLY.get(c, c) for c in self.columns}))
    def _friendly_repr(self): return _orig_repr(self.rename(columns={c: FRIENDLY.get(c, c) for c in self.columns}))
    _friendly_html._gbsa_wrapped = _friendly_repr._gbsa_wrapped = True
    pd.DataFrame._repr_html_, pd.DataFrame.__repr__ = _friendly_html, _friendly_repr

SELECTED_COMBO = "igb2_di4_salt0.15_st0.0072"   # the locked best-of-48 physics combo (see notebook 03)

def _figs():
    """Export every figure this notebook created to figures/ (the last-cell convention)."""
    for i, fig in enumerate(_CREATED_FIGS, start=1):
        fig.savefig(FIGURES / f"{NB_STEM}_fig{i}.png")
    print("exported", len(_CREATED_FIGS), "figure(s) to figures/")


## How much does the GBSA physics move the ranking?

**What we do.** Ask how much of the ranking-quality variance each MM-GBSA physics factor (`igb`, `intdiel`, `saltcon`, `surften`) explains. This is the check that decides whether we can safely lock one combo for validation.

**How we do it.** One-way variance decomposition (η² = between-level SS over total SS) of each factor on Kendall τ and on BEDROC, pooled across the 8 covered targets. We also report η² on BEDROC after blocking on target — that strips the between-target variance that would otherwise dominate.

Two steps. (1) Raw ΔG across all 48 combos, then per-(target, combo) metrics, then η² per factor. (2) η² becomes the importance plot.


**Step 1 — raw data to table.** Per-(target, combo) τ and BEDROC from raw. Parse the four factor levels out of each combo name. Take one-way η². Matches `data/derived/physics_factor_importance.csv`.


In [ ]:
import re
gbsa = load("gbsa_dG_raw"); meta = load("metadata")
merged = gbsa.merge(meta, on=["complex_id", "target"])

def parse_combo(combo):
    igb, di, salt, st = re.match(r"igb(\d+)_di(\d+)_salt([0-9.]+)_st([0-9.]+)", combo).groups()
    return {"igb": igb, "intdiel": di, "saltcon": salt, "surften": st}

metric_rows = []
for (target, combo), ligands in merged.groupby(["target", "combo"]):
    ligands = ligands.dropna(subset=["pchembl", "is_active", "mean_dG_kcalmol"])
    labels = ligands.is_active.astype(int).to_numpy()
    if labels.sum() == 0 or labels.sum() == len(labels):
        continue
    row = {"target": target, "combo": combo,
           "tau": metrics.kendall_tau(ligands.mean_dG_kcalmol.to_numpy(), ligands.pchembl.to_numpy()),
           "bedroc": metrics.bedroc(-ligands.mean_dG_kcalmol.to_numpy(), labels)}
    row.update(parse_combo(combo)); metric_rows.append(row)
combo_metrics = pd.DataFrame(metric_rows)

def eta2(frame, factor, response):
    grand = frame[response].mean(); ss_total = ((frame[response] - grand) ** 2).sum()
    ss_between = sum(len(g) * (g[response].mean() - grand) ** 2 for _, g in frame.groupby(factor))
    return ss_between / ss_total if ss_total > 0 else np.nan

def eta2_target_blocked(frame, factor, response):   # remove between-target variance first
    centred = frame.copy()
    centred[response] = centred[response] - centred.groupby("target")[response].transform("mean")
    return eta2(centred, factor, response)

FACTORS = ["igb", "intdiel", "saltcon", "surften"]
# Build at full precision, round only for display. An earlier version rounded inside the
# constructor, so the saved file inherited 3-dp display rounding (eta2 0.0012 -> 0.001).
importance_full = pd.DataFrame({"factor": FACTORS,
    "eta2_tau":    [eta2(combo_metrics, f, "tau") for f in FACTORS],
    "eta2_bedroc": [eta2(combo_metrics, f, "bedroc") for f in FACTORS],
    "eta2_bedroc_target_blocked": [eta2_target_blocked(combo_metrics, f, "bedroc") for f in FACTORS]})
importance = importance_full.round(4)

# Persist the eta-squared table this notebook already builds. Until round 6 this table was READ by verify.py and by
# later notebooks but WRITTEN by nothing: deleting data/derived/ and re-running
# the reading order did not bring it back. Found by our own lineage audit.
# Save FULL precision. `importance` is rounded for display; writing the rounded
# frame silently truncated eta-squared from 4 dp to 3 dp.
importance_full.to_csv(DERIVED / "physics_factor_importance.csv", index=False)
importance

**Step 2 — table to plot.** η² per factor on pooled τ and on target-blocked BEDROC, against a 5% reference.


In [ ]:
# LOG axis, because the four η² span five orders of magnitude (2.6e-7 to 0.16). On a linear
# axis `saltcon` drew as literally zero pixels on both bars and `surften`'s gold bar was
# absent -- two of four rows read as missing data rather than as small effects, which is
# the opposite of this figure's message (referee, five rounds). Every bar now also carries
# its value, so nothing depends on reading a bar length.
# FULL PRECISION, not the display frame. `importance` is `importance_full.round(4)`, so
# saltcon's tau eta-squared (2.6e-7) and surften's (2.1e-5) both round to EXACTLY 0.0 --
# which is why the gold bars for those two rows were literally absent for five rounds, and
# why a log axis alone would not have fixed it: a zero cannot be drawn on a log scale. The
# rounded frame is for display; every figure and every printed value now comes from the
# unrounded one. (Found by reading this cell's own output after fixing the axis.)
ordered = importance_full.sort_values("eta2_bedroc_target_blocked")
ys = np.arange(len(ordered)); bar_h = 0.38
_FLOOR = 1e-7                      # left edge; every η² here is above it
fig, ax = plt.subplots(figsize=(9.2, 4.2))
_b1 = ax.barh(ys + bar_h/2, ordered.eta2_bedroc_target_blocked * 100, bar_h,
              left=_FLOOR*100, color=NAVY, label="BEDROC (target-blocked)")
_b2 = ax.barh(ys - bar_h/2, ordered.eta2_tau * 100, bar_h,
              left=_FLOOR*100, color=GOLD, label="τ (pooled)")
ax.set_xscale("log")
ax.set_xlim(_FLOOR*100, 100)
for _bars, _vals in ((_b1, ordered.eta2_bedroc_target_blocked), (_b2, ordered.eta2_tau)):
    for _r, _v in zip(_bars, _vals):
        ax.text(_v*100*1.35, _r.get_y() + _r.get_height()/2,
                f"{_v*100:.4g} %", va="center", fontsize=7.5, color=GREYD)
ax.axvline(5, color=GREYD, lw=1.2, ls="--")
ax.text(5.5, 0.02, "5 % of variance", color=GREYD, fontsize=9, transform=ax.get_xaxis_transform(),
        va="bottom", bbox=dict(boxstyle="round,pad=0.2", fc=CREAM, ec="none", alpha=0.9))
ax.set_yticks(ys); ax.set_yticklabels(ordered.factor)
ax.set_xlabel("% of variance explained (η²) — log scale")
ax.legend(fontsize=9, frameon=False, loc="lower right")
ax.spines[["top", "right"]].set_visible(False); fig.tight_layout(); plt.show()

print("η² by factor, both responses (log axis above; the range is 5 orders of magnitude):")
for _r in ordered.sort_values("eta2_bedroc_target_blocked", ascending=False).itertuples():
    print(f"  {_r.factor:9s} BEDROC (target-blocked) {_r.eta2_bedroc_target_blocked*100:9.4g} %"
          f"   τ (pooled) {_r.eta2_tau*100:9.4g} %")
print("Only `intdiel` exceeds the 5 % line, and only on the blocked BEDROC response.")
_small = ordered.set_index("factor")
print(f"`saltcon` and `surften` are not zero -- on the blocked BEDROC response they are "
      f"{_small.loc['saltcon','eta2_bedroc_target_blocked']*100:.2g} % and "
      f"{_small.loc['surften','eta2_bedroc_target_blocked']*100:.2g} % -- but they are far")
print("below anything this design could distinguish from noise. They are drawn, not omitted,")
print("because a missing bar and a tiny bar mean different things.")


**Verdict.** Every physics factor explains **< 5% of ranking variance on pooled τ**. Target identity dominates. The biggest physics lever is `intdiel`, reaching η² ≈ 0.16 on BEDROC once we block on target. Because the result does not hinge on fine physics tuning, we can lock one combo (`igb2_di4_salt0.15_st0.0072`) for validation without cherry-picking.


### Same picture on BEDROC — does early enrichment also track the target?

The bars above pool the factors. The map below shows the raw surface. Every one of 48 physics combos × every covered target, both metrics side by side. `locked_settings.csv` names **BEDROC α=20** as the primary metric, so the τ panel alone does not settle the lock question. This one does.

Read by direction of banding. Vertical stripes mean the target sets the metric and combo choice barely moves it (safe to lock one combo). Horizontal stripes would mean combo drives the result (locking would be cherry-picking).

Different colour scales on purpose. τ is signed (−1 to +1, diverging around 0; negative = anti-predictive). BEDROC is bounded (0 to 1, sequential). Forcing one scale onto both would misstate τ.


In [ ]:
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.patches import Rectangle

LOCKED = "igb2_di4_salt0.15_st0.0072"   # data/derived/locked_settings.csv

_tau_p = combo_metrics.pivot(index="combo", columns="target", values="tau")
_bed_p = combo_metrics.pivot(index="combo", columns="target", values="bedroc")
_order = sorted(_tau_p.index)                      # igb -> intdiel -> salt -> surften
_tau_p, _bed_p = _tau_p.loc[_order], _bed_p.loc[_order]

# --- how much of each metric is target vs combo? (one-way eta^2 on the long table)
_var = pd.DataFrame({
    "metric": ["tau", "BEDROC"],
    "eta2_target": [eta2(combo_metrics, "target", "tau"),
                    eta2(combo_metrics, "target", "bedroc")],
    "eta2_combo":  [eta2(combo_metrics, "combo", "tau"),
                    eta2(combo_metrics, "combo", "bedroc")],
}).round(4)
_var["target_over_combo"] = (_var.eta2_target / _var.eta2_combo).round(1)
display(_var)

_div = LinearSegmentedColormap.from_list("navy_gold", [NAVY, "#f2f0e6", GOLD])
_seq = LinearSegmentedColormap.from_list("cream_navy", ["#f7f5ec", GOLD, NAVY])

fig, axes = plt.subplots(1, 2, figsize=(13.5, 11), sharey=True)
for ax, (_tab, _ttl, _cm, _nrm, _lab) in zip(axes, [
        (_tau_p, "(A) Kendall τ  (−ΔG vs pKi)", _div,
         TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1), "Kendall τ"),
        (_bed_p, "(B) BEDROC (α=20)  — the primary metric", _seq, None, "BEDROC")]):
    _im = ax.imshow(_tab.values, aspect="auto", cmap=_cm, norm=_nrm,
                    **({} if _nrm else {"vmin": 0, "vmax": 1}))
    ax.set_xticks(range(_tab.shape[1]))
    ax.set_xticklabels(_tab.columns, rotation=45, ha="right", fontsize=9)
    # NOTE: this notebook suppresses Axes.set_title/Figure.suptitle by design
    # (publication style -- markdown carries descriptions). Panel labels therefore
    # go in as in-axes text, which survives that suppression.
    ax.text(0.0, 1.012, _ttl, transform=ax.transAxes, ha="left", va="bottom",
            fontsize=11, weight="bold", color=NAVY)
    ax.set_xlabel("target")
    _r = list(_tab.index).index(LOCKED)
    ax.add_patch(Rectangle((-0.5, _r - 0.5), _tab.shape[1], 1, fill=False,
                           edgecolor=GOLD, lw=2.2, zorder=5))
    fig.colorbar(_im, ax=ax, fraction=0.035, pad=0.02, label=_lab)

axes[0].set_yticks(range(len(_order)))
axes[0].set_yticklabels(_order, fontsize=7, family="monospace")
for _t in axes[0].get_yticklabels():
    if _t.get_text() == LOCKED:
        _t.set_color(GOLD); _t.set_fontweight("bold")
axes[0].set_ylabel("physics combo (48)")
fig.tight_layout(rect=[0, 0, 1, 0.975])
plt.show()

**What the map shows.** Both panels band **vertically**. Columns (targets) differ strongly, rows (combos) barely. The η² table above puts a number on it — target identity explains many times more variance than combo choice, on both metrics. The gold row is GBSA-locked (`igb2_di4_salt0.15_st0.0072`). It sits in the same band as its neighbours rather than standing out. That is the condition for locking without cherry-picking.

**But BEDROC is looser.** The combo term is larger on BEDROC than on τ, consistent with `intdiel` reaching η² ≈ 0.16 on BEDROC once blocked. Early enrichment reads only the top of the list, so it is noisier and more sensitive to physics than a whole-list rank correlation. Locking one combo still holds — target dominates by a wide margin — but the claim "physics barely moves the ranking" is **weaker on BEDROC than on τ**. State it that way, not quoted from τ alone.


## Export figures
Save this notebook's figures to `figures/`.


In [ ]:
_figs()


In [ ]:
FIGURE_CAPTIONS = {
    '05_physics_importance_fig1.png':
        'η² of the four MM-GBSA physics parameters on target-blocked BEDROC (navy) and pooled Kendall τ (gold). Log x-axis, because the eight values span five orders of magnitude; every bar carries its value. Only intdiel exceeds the 5 % line, and only on the blocked BEDROC response.',
    '05_physics_importance_fig2.png':
        "Two heat maps over the full 48 combos x 9 targets grid — (A) Kendall tau on a diverging scale centred at 0, (B) BEDROC on a sequential scale — with the locked combo's row outlined. Target explains far more variance than combo does on both metrics, which is the notebook's point: the parameter response is target-specific, not uniform.",
}

# ---- captions, keyed by FILENAME ----------------------------------------------------
# The premise printed at the top of every notebook is that suppressed in-figure titles are
# carried by figures/CAPTIONS.md instead. That premise has been false twice: first no caption
# file existed at all, then titles were captured into a list nothing read. The third failure
# was subtler and is fixed here -- the file recorded TITLES but not FILENAMES, so a reader
# holding a PNG could not find its caption, and most notebooks contributed nothing because
# their titles had already been deleted rather than suppressed. Every figure this notebook
# writes now gets a line naming the file; captured titles are appended where they exist.
# verify.py section 16 asserts the coverage, and it is the first check in this package that
# can fail because of a picture (referee, six rounds).
_cap = FIGURES / "CAPTIONS.md"
_mine = sorted(p for p in FIGURES.rglob("*.png") if p.name.startswith(NB_STEM + "_"))
_prev = _cap.read_text() if _cap.exists() else ""
_keep = [l for l in _prev.splitlines()
         if l.startswith("- ") and f"**{NB_STEM}**" not in l]
_lines = []
for _p in _mine:
    _rel = _p.relative_to(FIGURES).as_posix()
    # match on the full basename first, then on the suffix after NB_STEM, because
    # notebooks 02-07 export as {stem}_fig{n} while 08-10 name each figure.
    _d = FIGURE_CAPTIONS.get(_p.name) or FIGURE_CAPTIONS.get(
        _p.stem.removeprefix(NB_STEM + "_"), "")
    _lines.append(f"- `{_rel}` — **{NB_STEM}** — {_d}" if _d
                  else f"- `{_rel}` — **{NB_STEM}** — NO CAPTION WRITTEN")

_lines += [f"- **{NB_STEM}** — suppressed title: {t}" for _, t in _SUPPRESSED_TITLES]
_cap.write_text("# Figure captions\n\nOne line per shipped figure, naming the file, plus any\n"
                "in-figure title suppressed for publication.\n\n"
                + "\n".join(sorted(set(_keep + _lines))) + "\n")
print(f"captions: {len(_mine)} figure(s) and {len(_SUPPRESSED_TITLES)} suppressed title(s) "
      f"recorded in {_cap.name}")
